In [1]:
# 패키지
import pandas as pd

In [2]:
# 데이터 로딩
customers_df = pd.read_csv(
    "../data/raw/olist_customers_dataset.csv"
)

orders_df = pd.read_csv(
    "../data/raw/olist_orders_dataset.csv"
)

order_items_df = pd.read_csv(
    "../data/raw/olist_order_items_dataset.csv"
)

payments_df = pd.read_csv(
    "../data/raw/olist_order_payments_dataset.csv"
)

products_df = pd.read_csv(
    "../data/raw/olist_products_dataset.csv"
)

category_translation_df = pd.read_csv(
    "../data/raw/product_category_name_translation.csv"
)

In [3]:
# 데이터 크기 확인
print("customers")
print(customers_df.shape)

print()

print("orders")
print(orders_df.shape)

print()

print("order_items")
print(order_items_df.shape)

print()

print("payments")
print(payments_df.shape)

print()

print("products")
print(products_df.shape)

print()

print("category_translation")
print(category_translation_df.shape)

customers
(99441, 5)

orders
(99441, 8)

order_items
(112650, 7)

payments
(103886, 5)

products
(32951, 9)

category_translation
(71, 2)


In [4]:
# 칼럼 확인
print(customers_df.columns.tolist())

print()
print(orders_df.columns.tolist())

print()
print(order_items_df.columns.tolist())

print()
print(payments_df.columns.tolist())

print()
print(products_df.columns.tolist())

['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']


In [5]:
# 고객 식별자 검증
print(
    "customer_id:",
    customers_df["customer_id"].nunique()
)

print(
    "customer_unique_id:",
    customers_df["customer_unique_id"].nunique()
)

print(
    "customer_id duplicates:",
    customers_df["customer_id"]
    .duplicated()
    .sum()
)

print(
    "customer_unique_id duplicates:",
    customers_df["customer_unique_id"]
    .duplicated()
    .sum()
)

customer_id: 99441
customer_unique_id: 96096
customer_id duplicates: 0
customer_unique_id duplicates: 3345


In [6]:
# 주문 key 검증
print(
    "orders order_id duplicate:",
    orders_df["order_id"]
    .duplicated()
    .sum()
)

print(
    "order_items duplicated order_id:",
    order_items_df["order_id"]
    .duplicated()
    .sum()
)

print(
    "payments duplicated order_id:",
    payments_df["order_id"]
    .duplicated()
    .sum()
)

orders order_id duplicate: 0
order_items duplicated order_id: 13984
payments duplicated order_id: 4446


In [7]:
# 주문 상태 Audit
print(
    orders_df["order_status"]
    .value_counts(
        dropna=False
    )
)

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [8]:
# 날짜 Audit
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for column in date_columns:
    orders_df[column] = pd.to_datetime(
        orders_df[column]
    )

print(
    orders_df[
        date_columns
    ]
    .isna()
    .sum()
)

print()

# 기간
print(
    orders_df["order_purchase_timestamp"].min()
)

print(
    orders_df["order_purchase_timestamp"].max()
)

print()

print(
    orders_df["order_approved_at"].min()
)

print(
    orders_df["order_approved_at"].max()
)

order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

2016-09-04 21:15:19
2018-10-17 17:30:18

2016-09-15 12:16:38
2018-09-03 17:40:06


In [9]:
# 승인 시점 결측 확인
approved_missing_df = (
    orders_df[
        orders_df["order_approved_at"].isna()
    ]
)

print(
    approved_missing_df[
        "order_status"
    ]
    .value_counts(
        dropna=False
    )
)

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64
